# Bước 0: Trích Xuất Mở Rộng Dữ Liệu HRV (Multi-Scale Raw Signal Feature Extraction)
Notebook này xử lý trực tiếp từ **dữ liệu sóng điện tim thô (RAW)** cho tập dữ liệu **MIMIC-III** (`data/raw/mimic_perform/ppg_af_dataset.csv` - 5.25 triệu điểm dữ liệu sóng).

Thực hiện trích xuất theo 4 quy mô dung lượng bằng kỹ thuật **Sliding Window**:
1. **`mimic_features_1360.csv`**: Cửa sổ 30s, bước trượt 30s (Không chồng lấp - 1,360 mẫu).
2. **`mimic_features_4083.csv`**: Cửa sổ 30s, bước trượt 10s (Trượt 66% - 4,083 mẫu - Mặc định).
3. **`mimic_features_8165.csv`**: Cửa sổ 30s, bước trượt 5s (Trượt 83% - 8,165 mẫu).
4. **`mimic_features_16358.csv`**: Cửa sổ 30s, bước trượt 2.5s (Trượt 91% - 16,358 mẫu).

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
from scipy.fft import rfft, rfftfreq
import warnings
warnings.filterwarnings('ignore')

# Thiết lập thư mục đầu ra data/features/
features_dir_candidates = ['../../data/features', '../data/features', 'data/features']
features_dir = next((d for d in features_dir_candidates if os.path.exists(os.path.dirname(d))), '../../data/features')
os.makedirs(features_dir, exist_ok=True)
print(f'✅ Thư mục đầu ra được thiết lập: {features_dir}')

### 1. Thuật toán trích xuất 16 chỉ số HRV y tế từ sóng thô (Nội suy 4Hz)

In [ ]:
def extract_hrv_from_window(signal, sampling_rate=125):
    min_dist = int(sampling_rate * 0.4)
    height_thresh = np.mean(signal) + 0.3 * np.std(signal)
    peaks, _ = find_peaks(signal, distance=min_dist, height=height_thresh)
    
    rr_intervals = np.diff(peaks) / float(sampling_rate)
    if len(rr_intervals) < 3:
        return None
    
    rr_ms = rr_intervals * 1000.0
    diff_rr = np.diff(rr_ms)
    
    mean_nn = np.mean(rr_ms)
    hr_mean = 60000.0 / mean_nn if mean_nn > 0 else 0.0
    sdnn = np.std(rr_ms)
    rmssd = np.sqrt(np.mean(diff_rr**2)) if len(diff_rr) > 0 else 0.0
    nn50 = int(np.sum(np.abs(diff_rr) > 50))
    pnn50 = float((nn50 / len(diff_rr)) * 100.0) if len(diff_rr) > 0 else 0.0
    cv = sdnn / mean_nn if mean_nn > 0 else 0.0
    
    # Nội suy chuỗi RR ở tần số 4Hz để tính phổ LF/HF
    time_rr = np.cumsum(rr_intervals)
    time_4hz = np.arange(time_rr[0], time_rr[-1], 0.25)
    if len(time_4hz) > 8:
        rr_4hz = np.interp(time_4hz, time_rr, rr_ms)
        mean_4hz = np.mean(rr_4hz)
        N = len(rr_4hz)
        yf = (np.abs(rfft(rr_4hz - mean_4hz))**2) / N
        xf = rfftfreq(N, 0.25)
        
        lf_band = (xf >= 0.04) & (xf < 0.15)
        hf_band = (xf >= 0.15) & (xf < 0.40)
        
        lf = float(np.sum(yf[lf_band])) if np.any(lf_band) else 0.0
        hf = float(np.sum(yf[hf_band])) if np.any(hf_band) else 0.0
        total_power = float(np.sum(yf))
        lf_hf_ratio = float(lf / hf) if hf > 0 else 0.0
        lf_norm = float((lf / (lf + hf + 1e-6)) * 100.0)
        hf_norm = float((hf / (lf + hf + 1e-6)) * 100.0)
    else:
        lf = hf = total_power = lf_hf_ratio = lf_norm = hf_norm = 0.0
    
    # Poincaré Metrics
    sd1 = np.sqrt(0.5 * np.var(diff_rr)) if len(diff_rr) > 0 else 0.0
    sd2 = np.sqrt(max(0, 2 * np.var(rr_ms) - 0.5 * np.var(diff_rr))) if len(diff_rr) > 0 else 0.0
    samp_en = float(np.std(diff_rr) / (sdnn + 1e-6))
    
    return {
        'HR_mean': hr_mean,
        'Mean_NN': mean_nn,
        'SDNN': sdnn,
        'RMSSD': rmssd,
        'NN50': nn50,
        'pNN50': pnn50,
        'CV': cv,
        'LF': lf,
        'HF': hf,
        'Total_Power': total_power,
        'LF_HF_Ratio': lf_hf_ratio,
        'LF_norm': lf_norm,
        'HF_norm': hf_norm,
        'SD1': sd1,
        'SD2': sd2,
        'SampEn': samp_en
    }
print('✅ Thuật toán trích xuất 16 chỉ số HRV sẵn sàng!')

### 2. Trích xuất cả 4 quy mô dung lượng MIMIC-III (1360, 4083, 8165, 16358)

In [ ]:
raw_mimic_candidates = ['../../data/raw/mimic_perform/ppg_af_dataset.csv', '../data/raw/mimic_perform/ppg_af_dataset.csv', 'data/raw/mimic_perform/ppg_af_dataset.csv']
raw_mimic_path = next((p for p in raw_mimic_candidates if os.path.exists(p)), None)

if raw_mimic_path:
    print(f'⚡ Đang đọc dữ liệu thô MIMIC-III từ: {raw_mimic_path}...')
    df_raw = pd.read_csv(raw_mimic_path)
    window_size = 3750  # 30s @ 125Hz
    
    scales = {
        '1360': 3750,   # 30s step
        '4083': 1250,   # 10s step
        '8165': 625,    # 5s step
        '16358': 312    # 2.5s step
    }
    
    for tag, step_size in scales.items():
        records = []
        for i in range(0, len(df_raw) - window_size, step_size):
            sub = df_raw.iloc[i : i + window_size]
            status = sub['status'].mode()[0]
            feat = extract_hrv_from_window(sub['ecg'].values, sampling_rate=125)
            if feat:
                feat['status'] = status
                records.append(feat)
        
        df_feat = pd.DataFrame(records)
        out_file = os.path.join(features_dir, f'mimic_features_{tag}.csv')
        df_feat.to_csv(out_file, index=False)
        print(f'🎉 [{tag} Mẫu] Đã lưu vào: {out_file} ({df_feat.shape[0]} mẫu)')
        if tag == '4083':
            df_feat.to_csv(os.path.join(features_dir, 'mimic_features.csv'), index=False)
else:
    print('❌ Không tìm thấy file dữ liệu thô MIMIC!')